In [0]:
from pyspark.sql.functions import *

In [0]:
df_dept_hospA = spark.read.option('header', True).parquet('abfss://bronze@adlshosptial.dfs.core.windows.net/EMR/HospitalA/departments')

df_dept_hospB = spark.read.option('header', True).parquet('abfss://bronze@adlshosptial.dfs.core.windows.net/EMR/HospitalB/departments')

df_dept = df_dept_hospA.unionByName(df_dept_hospB)

df_dept = df_dept.withColumn('datasource',regexp_replace(col('datasource'),'-','_'))

df_dept = df_dept.withColumn('Src_DeptID',col('DeptID'))\
            .withColumn('Dept_ID', concat(col('DeptID'),lit('_'),col('datasource')))\
            .drop('DeptID')

df_dept = df_dept.select('Src_DeptID','Dept_ID','Name','datasource')

df_dept.createOrReplaceTempView('Departments')

display(df_dept)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS silver.departments(
    Src_DeptID varchar(200),
    Dept_ID VARCHAR(200),
    Name VARCHAR(200),
    datasource VARCHAR(200),
    is_quarantined BOOLEAN
)
using DELTA;

In [0]:
%sql
TRUNCATE TABLE silver.departments;

INSERT INTO silver.departments(
    Src_DeptID,
    Dept_ID,
    Name,
    datasource,
    is_quarantined
    )
    SELECT
        Src_DeptID,
        Dept_ID,
        Name,
        datasource,
        CASE
            WHEN Dept_ID is NULL or Name is NULL THEN TRUE
            ELSE FALSE
        END
        AS is_quarantined
    FROM Departments;